In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from app.core.db import connect
from app.rag.search import search
from app.llm.client import check_ollama

print("Ollama running:", check_ollama())
print(connect().execute("SELECT source_type, count(*) AS chunks FROM chunks GROUP BY 1").fetchall())


def ask(question, region=None, k=4, min_score=-1.0, source_type=None):
    print(f"Q: {question}   (region: {region or 'all'})\n")
    hits = search(question, k=k, min_score=min_score, region=region, source_type=source_type)
    if not hits:
        print("No chunk is close enough.")
    for hit in hits:
        print(f"{hit.score:.3f}  [{hit.metadata['chunk_id']}]")
        print("       " + " ".join(hit.text.split())[:220] + "...\n")

## A question about the law

In [ ]:
ask("What does the law say about forcing a child into marriage?")

## A question about cases in one region

In [ ]:
ask("Is any child being kept from school to work?", region="northern")

## Only the policies

In [ ]:
ask("Can a child be beaten as punishment at school?", source_type="policy")

In [ ]:
ask("How many high risk cases are there? Can you give me a nuber?", region="upper_east")

# Chat with the assistant


In [ ]:
from app.tools.assistant import chat
from app.services import chat_store

region = "northern"
conversation = chat_store.create_conversation()
print("New conversation for region:", region)

In [ ]:
question = input("Please enter your question: ")

reply = chat(conversation, question, region)

if reply.needs_approval:
    print("\n" + reply.text)
    if input("Continue? (yes/no): ").strip().lower() in ("yes", "y"):
        reply = chat(conversation, question, region, approved=reply.needs_approval)

print("\nQuestion:", question)
print("\nAI CARE Assistant:", reply.with_sources())